In [2]:
from summer3 import lazyprops as slp
from summer3.examples import get_tb_cmap
import polars as pl
import numpy as np

In [3]:
cmap, strats = get_tb_cmap()#age=True, loc=256*256)

In [4]:
#def stratify_table(pt, query, strat):
#pt.filter((acc.age == "75") & (acc.history == "active"))

In [5]:
pt = slp.build_property_tables(cmap, cmap.compartments)

In [6]:
class PropertyCMap:
    def __init__(self, cmap, prop_table: slp.PropertyTable):
        self.cmap = cmap
        self.prop_table = prop_table

    @classmethod
    def from_cmap(cls, cmap):
        return cls(cmap, slp.build_property_tables(cmap, cmap.compartments))
    
    def filter(self, expr):
        #pt = self.prop_table
        #query_expr = expr.actualize(pt)

        #filtered_df = pt.table.filter(query_expr)

        #validity_s = filtered_df.select(~pl.all().eq(-1).all())

        #valid_columns = [c.name for c in validity_s if c[0]]
        #valid_columns = [k for k,v in validity.items() if v]

        #valid_df = filtered_df[valid_columns]

        # Pass valid_df to erase stratifications, or filtered_df to retain all
        new_pt = self.prop_table.filter(expr)

        return PropertyCMap(cmap[new_pt.table["index"].to_numpy()], new_pt)
    
    def get_accessors(self):
        from collections import namedtuple
        props = self.prop_table.properties
        keys = [k.name for k in props]
        return namedtuple("Accessor", keys)(*props.values())
        #return list(self.prop_table.properties.values())
    
    def __repr__(self):
        return f"[PCMap]{self.prop_table}"

In [7]:
pcmap = PropertyCMap.from_cmap(cmap)
acc = pcmap.get_accessors()
pt = pcmap.prop_table

In [ ]:
def valid_columns(df):
    validity_s = df.select(~pl.all().is_null().all())
    valid_columns = [c.name for c in validity_s if c[0] and (c.name != "index")]
    return valid_columns

def invalid_columns(df):
    invalidity_s = df.select(pl.all().is_null().all())
    invalid_columns = [c.name for c in invalidity_s if c[0]]
    return invalid_columns

def cat_counts(df, column):
    return df.group_by(column).len()#.rename({"len": "repetition_count"})

def repeat_on_other(df1, df2, column):
    # 1. Calculate value_counts for the 'category' column
    category_counts2 = df2.group_by(column).len().rename({"len": "repetition_count"})

    # 2. Join with the original DataFrame
    df1_with_counts = df1.join(category_counts2, on=column, how="left")

    # 3. Apply repeat_by and flatten
    repeated_df = df1_with_counts.select(
        pl.all().repeat_by(pl.col("repetition_count")).flatten()
    ).drop("repetition_count") # Drop the repetition_count column if not needed

    return repeated_df

def relabel_props(df, prepend):
    renamed = df.rename({col:f"{prepend}__{col}" for col in df.columns})
    return renamed

def create_index(df):
    return df.with_columns(index=np.arange(len(df)))

def expr_from_dict(prop_dict):
    expr = None
    for col,val in prop_dict.items():
        if expr is None:
            expr = pl.col(col) == val
        else:
            expr = expr & (pl.col(col) == val)
    return expr

def repeat_from_counts(fs, basis_columns, ref_src=True):
    if ref_src:
        ref = fs.src_df
        target = fs.dest_df
        extras = fs.created_props
    else:
        ref = fs.dest_df
        target = fs.src_df
        extras = fs.erased_props

    cc = target.group_by(basis_columns + extras).len().group_by(basis_columns).len()



    accum = []
    for row in cc.iter_rows(named=True):
        length = row.pop("len")

        #expr = None
        #for col,val in row.items():
        #    if expr is None:
        #        expr = pl.col(col) == val
        #    else:
        #        expr = expr & (pl.col(col) == val)

        expr = expr_from_dict(row)

        basis_df = ref.filter(expr)
        adf_exp = pl.concat([basis_df]*length)
        accum.append(adf_exp)

    return pl.concat(accum)

In [9]:
from itertools import product

In [10]:
class FlowSpec:
    def __init__(self, srcq: slp.LazyExpr, destq: slp.LazyExpr, pt: slp.PropertyTable):
        self.srcq = srcq
        self.destq = destq
        self.pt = pt

        self.src_table = pt.filter(srcq)
        self.dest_table = pt.filter(destq)

        self.src_df = self.src_table.table
        self.dest_df = self.dest_table.table

        self.src_props = src_props = valid_columns(self.src_df)
        self.dest_props = dest_props = valid_columns(self.dest_df)

        self.common_props = list(set(src_props).intersection(set(dest_props)))

        self.erased_props = list(set(src_props).difference(set(dest_props)))
        self.created_props = list(set(dest_props).difference(set(src_props)))

    def get_policy_tables(self):
        props_matched = [] # A->A
        props_moved = [] # Nothing in common (moved)
        props_to_map = []

        for column in self.common_props:
            src_props = self.src_df[column].unique()
            dest_props = self.dest_df[column].unique()

            if (src_props == dest_props).all():
                props_matched.append(column)
            elif (src_props != dest_props).all():
                props_moved.append(column)
            else:
                props_to_map.append(column)        

        return {"matched": props_matched, "moved": props_moved, "mapped": props_to_map}
    
    def realise_mapping(self):

        policy = self.get_policy_tables()

        #if len(policy["matched"]) > 1:
        #    raise Exception("Only single match supported for now")
        basis = policy["matched"]
        src_mapped = create_index(relabel_props(repeat_from_counts(self, basis, True), "source"))
        dest_mapped = create_index(relabel_props(repeat_from_counts(self, basis, False), "dest"))

        return src_mapped.join(dest_mapped, on="index")
        

In [13]:
infection = FlowSpec(acc.history == "naive", acc.history == "early", pt)
progression = FlowSpec(acc.history == "early", acc.history == "active", pt)

In [14]:
#progression.realise_mapping()

In [57]:
dest_prop_combos = product(*[progression.dest_df[prop].unique().to_list() for prop in progression.created_props])
dpcl = list(dest_prop_combos)
for dpci in dpcl:
    prop_dict = expr_from_dict({prop:val for (prop,val) in zip(progression.created_props, dpci)})
    print(len(progression.dest_df.filter(prop_dict)))


196608
196608
196608
196608


In [69]:
dest_prop_combos = product(*[progression.src_df[prop].unique().to_list() for prop in progression.erased_props])
dpcl = list(dest_prop_combos)
for dpci in dpcl:
    prop_dict = expr_from_dict({prop:val for (prop,val) in zip(progression.erased_props, dpci)})
    print(len(progression.src_df.filter(prop_dict)))


196608
196608
196608


In [53]:
def perfect_tiling(df: pl.DataFrame, ref_col: str) -> bool:
    """Check if a DataFrame contains only repeated blocks, that tile exactly
    an integer number of times over the 'axis' specified by ref_col, such
    that the 1d array represented by this table could be represented as 2d
    by unwrapping over this column

    Args:
        df: The DataFrame to check
        ref_col: _description_

    Returns:
        Tiling status
    """
    
    uvals = df.select(pl.col(ref_col)).unique().to_numpy()[:,0]
    ref_block = df.filter(pl.col(ref_col) == uvals[0])
    ref_block = ref_block.drop(invalid_columns(ref_block) + ["index", ref_col])

    a = ref_block
    b = df.drop(invalid_columns(df) + ["index", ref_col])

    if set(a.columns) != set(b.columns):
        return False

    comp_tiled = pl.DataFrame(np.tile(a.to_numpy().T,len(b)//len(a)).T, schema=a.columns)

    return (comp_tiled==b).select(pl.all_horizontal(pl.all()).all()).item()

In [65]:
subset = pt.table.sort("age_0").filter(pl.col("history_0")==0)

In [73]:
xjs = xj[subset["index"].to_numpy()]

In [ ]:
def col_shapes(df):
    return df.select(pl.all().n_unique()).to_dict(as_series=False)

In [90]:
subset.select(pl.all().n_unique())["history_0"].item()

1

In [81]:
col_shapes(subset)

history_0,age_0,loc_0,early_tb_0,inf_status_0,clin_status_0,index
u32,u32,u32,u32,u32,u32,u32
1,3,65536,1,1,1,196608
1,3,65536,1,1,1,196608
1,3,65536,1,1,1,196608
1,3,65536,1,1,1,196608
1,3,65536,1,1,1,196608
…,…,…,…,…,…,…
1,3,65536,1,1,1,196608
1,3,65536,1,1,1,196608
1,3,65536,1,1,1,196608


In [66]:
perfect_tiling(subset, "age_0")

True

In [58]:
perfect_tiling(pt.table.sort("loc_0"), "loc_0")

True

In [16]:
progression.src_df

history_0,age_0,loc_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64,i64
1,0,0,0,null,null,196608
1,0,0,1,null,null,196609
1,0,0,2,null,null,196610
1,0,1,0,null,null,196611
1,0,1,1,null,null,196612
…,…,…,…,…,…,…
1,2,65534,1,null,null,786427
1,2,65534,2,null,null,786428
1,2,65535,0,null,null,786429


In [17]:
from jax import numpy as jnp, jit, lax, Array

In [18]:
xj = jnp.array(pt.table["index"],dtype=jnp.float64)

In [20]:
idx = progression.src_df["index"].to_numpy()#.astype(np.int32)

In [197]:
progression.src_df

history_0,age_0,loc_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64,i64
1,0,0,0,null,null,196608
1,0,0,1,null,null,196609
1,0,0,2,null,null,196610
1,0,1,0,null,null,196611
1,0,1,1,null,null,196612
…,…,…,…,…,…,…
1,2,65534,1,null,null,786427
1,2,65534,2,null,null,786428
1,2,65535,0,null,null,786429


In [41]:
perfect_tiling(progression.src_df.sort("early_tb_0"), "early_tb_0")

True

In [186]:
pl.DataFrame(np.tile(a.to_numpy().T,len(b)//len(a)).T, schema=a.columns)

history_0,age_0,early_tb_0
i64,i64,i64
1,0,0
1,0,1
1,0,2
1,1,0
1,1,1
…,…,…
1,1,1
1,1,2
1,2,0


history_0,age_0,early_tb_0
i64,i64,i64
1,0,0
1,0,1
1,0,2
1,1,0
1,1,1
…,…,…
1,1,1
1,1,2
1,2,0


In [ ]:
(a==b).select(pl.all_horizontal(pl.all()).all()).item()

True

In [85]:
256*256

65536

In [109]:
(progression.src_df.filter(pl.col("loc_0")==0) == progression.src_df.filter(pl.col("loc_0")==1)).drop(["loc_0","index"]).select(pl.all().all()).select(pl.all_horizontal(pl.all()))

history_0
bool
true


In [68]:
progression.dest_df["inf_status_0"].is_null().any()

False

In [61]:
prop_dict

<Expr ['[(col("inf_status_0")) == (dyn…'] at 0x229F1341350>

In [56]:
progression.src_df

history_0,age_0,loc_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64,i64
1,0,0,0,null,null,196608
1,0,0,1,null,null,196609
1,0,0,2,null,null,196610
1,0,1,0,null,null,196611
1,0,1,1,null,null,196612
…,…,…,…,…,…,…
1,2,65534,1,null,null,786427
1,2,65534,2,null,null,786428
1,2,65535,0,null,null,786429


In [53]:
progression.dest_df.filter(prop_dict)

history_0,age_0,loc_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64,i64
2,0,0,null,1,1,786435
2,0,1,null,1,1,786439
2,0,2,null,1,1,786443
2,0,3,null,1,1,786447
2,0,4,null,1,1,786451
…,…,…,…,…,…,…
2,2,65531,null,1,1,1572847
2,2,65532,null,1,1,1572851
2,2,65533,null,1,1,1572855


In [10]:
infection.get_policy_tables()

{'matched': ['loc_0', 'age_0'], 'moved': ['history_0'], 'mapped': []}

In [11]:
infection.realise_mapping()

Exception: Only single match supported for now

In [11]:
for p in progression.erased_props:
    print(p, cat_counts(progression.src_df, p))

early_tb_0 shape: (3, 2)
┌────────────┬─────┐
│ early_tb_0 ┆ len │
│ ---        ┆ --- │
│ i64        ┆ u32 │
╞════════════╪═════╡
│ 0          ┆ 3   │
│ 1          ┆ 3   │
│ 2          ┆ 3   │
└────────────┴─────┘


In [26]:
src_mapped = create_index(relabel_props(repeat_from_counts(progression, "age_0", True), "source"))
dest_mapped = create_index(relabel_props(repeat_from_counts(progression, "age_0", False), "dest"))

src_mapped.join(dest_mapped, on="index")

source__history_0,source__early_tb_0,source__inf_status_0,source__age_0,source__clin_status_0,source__index,index,dest__history_0,dest__early_tb_0,dest__inf_status_0,dest__age_0,dest__clin_status_0,dest__index
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,0,null,0,null,3,0,2,null,0,0,0,12
1,1,null,0,null,6,1,2,null,0,0,1,13
1,2,null,0,null,9,2,2,null,1,0,0,18
1,0,null,0,null,3,3,2,null,1,0,1,19
1,1,null,0,null,6,4,2,null,0,0,0,12
…,…,…,…,…,…,…,…,…,…,…,…,…
1,1,null,2,null,8,31,2,null,1,2,1,23
1,2,null,2,null,11,32,2,null,0,2,0,16
1,0,null,2,null,5,33,2,null,0,2,1,17


In [32]:
src_mapped = create_index(relabel_props(repeat_from_counts(progression, "age_0", True), "source"))
dest_mapped = create_index(relabel_props(repeat_from_counts(progression, "age_0", False), "dest"))

src_mapped.join(dest_mapped, on="index").sort("source__index").filter((pl.col("source__age_0")==0) & (pl.col("source__early_tb_0") == 0))

source__history_0,source__early_tb_0,source__inf_status_0,source__age_0,source__clin_status_0,source__index,index,dest__history_0,dest__early_tb_0,dest__inf_status_0,dest__age_0,dest__clin_status_0,dest__index
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,0,null,0,null,3,0,2,null,0,2,0,16
1,0,null,0,null,3,3,2,null,1,2,1,23
1,0,null,0,null,3,6,2,null,1,2,0,22
1,0,null,0,null,3,9,2,null,0,2,1,17


In [ ]:

def repeat_on_other(df1, df2, column):
    # 1. Calculate value_counts for the 'category' column
    category_counts2 = df2.group_by(column).len().rename({"len": "repetition_count"})

    # 2. Join with the original DataFrame
    df1_with_counts = df1.join(category_counts2, on=column, how="left")

    # 3. Apply repeat_by and flatten
    repeated_df = df1_with_counts.select(
        pl.all().repeat_by(pl.col("repetition_count")).flatten()
    ).drop("repetition_count") # Drop the repetition_count column if not needed

    return repeated_df


In [ ]:
repeat_on_other(progression.src_df, progression_dest_df)

In [379]:
for p in progression.created_props:
    print(p, cat_counts(progression.dest_df, p))

inf_status_0 shape: (2, 2)
┌──────────────┬─────┐
│ inf_status_0 ┆ len │
│ ---          ┆ --- │
│ i64          ┆ u32 │
╞══════════════╪═════╡
│ 1            ┆ 6   │
│ 0            ┆ 6   │
└──────────────┴─────┘
clin_status_0 shape: (2, 2)
┌───────────────┬─────┐
│ clin_status_0 ┆ len │
│ ---           ┆ --- │
│ i64           ┆ u32 │
╞═══════════════╪═════╡
│ 0             ┆ 6   │
│ 1             ┆ 6   │
└───────────────┴─────┘


In [ ]:
cat_counts(progression.src_df, "early_tb_0")

In [ ]:
cat_counts(progression.src_df, "early_tb_0")

early_tb_0,len
i64,u32
0,3
2,3
1,3


In [367]:
len(progression.src_df), len(progression.dest_df)

(9, 12)

In [ ]:
repeat_on_other()

In [358]:
infection.common_props

['history_0', 'age_0']

In [270]:
src_table = pt.filter(acc.history == "naive")
dest_table = pt.filter(acc.history == "early")

src_df = src_table.table
dest_df = dest_table.table

In [342]:
src_props = valid_columns(src_table)
dest_props = valid_columns(dest_table)

common_props = list(set(src_props).intersection(set(dest_props)))
common_props

['history_0', 'age_0']

In [326]:
# assumed_mapping: A->A

In [332]:
props_matched = [] # A->A
props_moved = [] # Nothing in common (moved)
props_to_map = []

for column in common_props:
    src_props = src_df[column].unique()
    dest_props = dest_df[column].unique()

    if (src_props == dest_props).all():
        props_matched.append(column)
    elif (src_props != dest_props).all():
        props_moved.append(column)
    else:
        props_to_map.append(column)

In [335]:
props_matched, props_moved, props_to_map

(['age_0'], ['history_0'], [])

In [338]:
src_df.group_by("age_0").len().rename({"len": "repetition_count"}), dest_df.group_by("age_0").len().rename({"len": "repetition_count"})

(shape: (3, 2)
 ┌───────┬──────────────────┐
 │ age_0 ┆ repetition_count │
 │ ---   ┆ ---              │
 │ i64   ┆ u32              │
 ╞═══════╪══════════════════╡
 │ 1     ┆ 1                │
 │ 0     ┆ 1                │
 │ 2     ┆ 1                │
 └───────┴──────────────────┘,
 shape: (3, 2)
 ┌───────┬──────────────────┐
 │ age_0 ┆ repetition_count │
 │ ---   ┆ ---              │
 │ i64   ┆ u32              │
 ╞═══════╪══════════════════╡
 │ 0     ┆ 3                │
 │ 2     ┆ 3                │
 │ 1     ┆ 3                │
 └───────┴──────────────────┘)

In [336]:
expanded_src = repeat_on_other(src_df, dest_df, "age_0")

shape: (3, 2)
┌───────┬──────────────────┐
│ age_0 ┆ repetition_count │
│ ---   ┆ ---              │
│ i64   ┆ u32              │
╞═══════╪══════════════════╡
│ 0     ┆ 3                │
│ 1     ┆ 3                │
│ 2     ┆ 3                │
└───────┴──────────────────┘


In [331]:
expanded_src[common_props] == dest_df[common_props]

history_0,age_0
bool,bool
false,true
false,true
false,true
false,true
false,true
false,true
false,true
false,true
false,true


In [311]:
dest_df.select(pl.col("age_0").value_counts())

age_0
struct[2]
"{1,3}"
"{0,3}"
"{2,3}"


In [278]:
src_df[list(common_props)], dest_df[list(common_props)]

(shape: (3, 2)
 ┌───────────┬───────┐
 │ history_0 ┆ age_0 │
 │ ---       ┆ ---   │
 │ i64       ┆ i64   │
 ╞═══════════╪═══════╡
 │ 0         ┆ 0     │
 │ 0         ┆ 1     │
 │ 0         ┆ 2     │
 └───────────┴───────┘,
 shape: (9, 2)
 ┌───────────┬───────┐
 │ history_0 ┆ age_0 │
 │ ---       ┆ ---   │
 │ i64       ┆ i64   │
 ╞═══════════╪═══════╡
 │ 1         ┆ 0     │
 │ 1         ┆ 0     │
 │ 1         ┆ 0     │
 │ 1         ┆ 1     │
 │ 1         ┆ 1     │
 │ 1         ┆ 1     │
 │ 1         ┆ 2     │
 │ 1         ┆ 2     │
 │ 1         ┆ 2     │
 └───────────┴───────┘)

In [217]:
src_df

history_0,age_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64
1,0,0,null,null,3
1,0,1,null,null,4
1,0,2,null,null,5
1,1,0,null,null,6
1,1,1,null,null,7
1,1,2,null,null,8
1,2,0,null,null,9
1,2,1,null,null,10
1,2,2,null,null,11


In [234]:
dest_table

PropertyTable
shape: (9, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ early     ┆ 0     ┆ incipient  ┆ null         ┆ null          ┆ 3     │
│ early     ┆ 0     ┆ contained  ┆ null         ┆ null          ┆ 4     │
│ early     ┆ 0     ┆ cleared    ┆ null         ┆ null          ┆ 5     │
│ early     ┆ 15    ┆ incipient  ┆ null         ┆ null          ┆ 6     │
│ early     ┆ 15    ┆ contained  ┆ null         ┆ null          ┆ 7     │
│ early     ┆ 15    ┆ cleared    ┆ null         ┆ null          ┆ 8     │
│ early     ┆ 75    ┆ incipient  ┆ null         ┆ null          ┆ 9     │
│ early     ┆ 75    ┆ contained  ┆ null         ┆ null          ┆ 10    │
│ early   

In [235]:
from copy import deepcopy

In [239]:
src_expanded = deepcopy(src_table)
src_df = src_table.table
src_expanded.table = src_df.with_columns(pl.col("age_0").repeat_by(3).alias("repeated")).explode("repeated").drop("repeated")

In [257]:
src_df.with_columns(
    pl.col("age_0").replace_strict(
        {0: 2, 1:2, 2: 4}).alias(
            "age_count")
        ).with_columns(
            pl.col("age_0").repeat_by(pl.col("age_count")).alias("age_repeated")
        ).explode("age_repeated").drop(["age_count","age_repeated"])

history_0,age_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64
0,0,null,null,null,0
0,0,null,null,null,0
0,1,null,null,null,1
0,1,null,null,null,1
0,2,null,null,null,2
0,2,null,null,null,2
0,2,null,null,null,2
0,2,null,null,null,2


In [244]:
src_expanded.table = src_df.with_columns(pl.col("age_0").repeat_by({0: 3}).alias("repeated")).explode("repeated").drop("repeated")

TypeError: cannot create expression literal for value of type dict.

Hint: Pass `allow_object=True` to accept any value and create a literal of type Object.

In [241]:
src_expanded

PropertyTable
shape: (9, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ naive     ┆ 0     ┆ null       ┆ null         ┆ null          ┆ 0     │
│ naive     ┆ 0     ┆ null       ┆ null         ┆ null          ┆ 0     │
│ naive     ┆ 0     ┆ null       ┆ null         ┆ null          ┆ 0     │
│ naive     ┆ 15    ┆ null       ┆ null         ┆ null          ┆ 1     │
│ naive     ┆ 15    ┆ null       ┆ null         ┆ null          ┆ 1     │
│ naive     ┆ 15    ┆ null       ┆ null         ┆ null          ┆ 1     │
│ naive     ┆ 75    ┆ null       ┆ null         ┆ null          ┆ 2     │
│ naive     ┆ 75    ┆ null       ┆ null         ┆ null          ┆ 2     │
│ naive   

In [243]:
relabel_props(src_expanded.table, "source")

source__history_0,source__age_0,source__early_tb_0,source__inf_status_0,source__clin_status_0,index
i64,i64,i64,i64,i64,i64
0,0,null,null,null,0
0,0,null,null,null,0
0,0,null,null,null,0
0,1,null,null,null,1
0,1,null,null,null,1
0,1,null,null,null,1
0,2,null,null,null,2
0,2,null,null,null,2
0,2,null,null,null,2


In [242]:
dest_table

PropertyTable
shape: (9, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ early     ┆ 0     ┆ incipient  ┆ null         ┆ null          ┆ 3     │
│ early     ┆ 0     ┆ contained  ┆ null         ┆ null          ┆ 4     │
│ early     ┆ 0     ┆ cleared    ┆ null         ┆ null          ┆ 5     │
│ early     ┆ 15    ┆ incipient  ┆ null         ┆ null          ┆ 6     │
│ early     ┆ 15    ┆ contained  ┆ null         ┆ null          ┆ 7     │
│ early     ┆ 15    ┆ cleared    ┆ null         ┆ null          ┆ 8     │
│ early     ┆ 75    ┆ incipient  ┆ null         ┆ null          ┆ 9     │
│ early     ┆ 75    ┆ contained  ┆ null         ┆ null          ┆ 10    │
│ early   

In [233]:
src_df[np.array([0,0,1])]

history_0,age_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64
0,0,null,null,null,0
0,0,null,null,null,0
0,1,null,null,null,1


In [227]:
src_df = src_table.table
src_df.with_columns(pl.col("age_0").repeat_by(3).alias("repeated")).explode("repeated").drop("repeated")

history_0,age_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64
0,0,null,null,null,0
0,0,null,null,null,0
0,0,null,null,null,0
0,1,null,null,null,1
0,1,null,null,null,1
0,1,null,null,null,1
0,2,null,null,null,2
0,2,null,null,null,2
0,2,null,null,null,2


In [220]:
valid_columns(dest_table)

['history_0', 'age_0', 'early_tb_0']

In [187]:
def relabel_props(df, prepend):
    renamed = df.rename({col:f"{prepend}__{col}" for col in df.columns if col != "index"})
    return renamed

In [188]:
relabel_props(src_table.table, "source")

source__history_0,source__age_0,source__early_tb_0,source__inf_status_0,source__clin_status_0,index
i64,i64,i64,i64,i64,i64
2,0,null,0,0,12
2,0,null,0,1,13
2,1,null,0,0,16
2,1,null,0,1,17
2,2,null,0,0,20
2,2,null,0,1,21


In [184]:
dest_table.table

history_0,age_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64
2,0,null,1,0,14
2,0,null,1,1,15
2,1,null,1,0,18
2,1,null,1,1,19
2,2,null,1,0,22
2,2,null,1,1,23


In [182]:
src_df = src_table.table.rename({col:f"source__{col}" for col in src_table.table.columns if col != "index"})

In [43]:
class PropertyData:
    def __init__(self, data, pt):
        self.data = data
        self.pt = pt

    def filter(self, expr):
        pass

In [91]:
pdata = PropertyData(jnp.linspace(0.0,1.0, 27), pcmap.prop_table)

In [172]:
pcmap.cmap[get_cat_indices(inf_cats, pt)[0]]

CompartmentContainer view of 0x2836293126416:
array([Compartment :[(Stratification: history, 'active'), (Stratification: age, '0'), (Stratification: inf_status, 'noninf'), (Stratification: clin_status, 'subclin')],
       Compartment :[(Stratification: history, 'active'), (Stratification: age, '0'), (Stratification: inf_status, 'noninf'), (Stratification: clin_status, 'clin')],
       Compartment :[(Stratification: history, 'active'), (Stratification: age, '15'), (Stratification: inf_status, 'noninf'), (Stratification: clin_status, 'subclin')],
       Compartment :[(Stratification: history, 'active'), (Stratification: age, '15'), (Stratification: inf_status, 'noninf'), (Stratification: clin_status, 'clin')],
       Compartment :[(Stratification: history, 'active'), (Stratification: age, '75'), (Stratification: inf_status, 'noninf'), (Stratification: clin_status, 'subclin')],
       Compartment :[(Stratification: history, 'active'), (Stratification: age, '75'), (Stratification: inf_stat

In [167]:
from summer3 import graph as s3g

In [ ]:
0,0,0,0
1,1,1,1


In [165]:
pdata.data.at[get_cat_indices(inf_cats, pt)].mul(np.array([(1.5,2.5)]).T)

Array([0.        , 0.03846154, 0.07692308, 0.11538462, 0.15384615,
       0.19230769, 0.23076923, 0.26923077, 0.30769231, 0.34615385,
       0.38461538, 0.42307692, 0.69230769, 0.75      , 1.34615385,
       1.44230769, 0.92307692, 0.98076923, 1.73076923, 1.82692308,
       1.15384615, 1.21153846, 2.11538462, 2.21153846, 0.92307692,
       0.96153846, 1.        ], dtype=float64)

In [94]:
pcmap.filter(acc.age == "15")

[PCMap]PropertyTable
shape: (9, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ naive     ┆ 15    ┆ null       ┆ null         ┆ null          ┆ 1     │
│ early     ┆ 15    ┆ incipient  ┆ null         ┆ null          ┆ 6     │
│ early     ┆ 15    ┆ contained  ┆ null         ┆ null          ┆ 7     │
│ early     ┆ 15    ┆ cleared    ┆ null         ┆ null          ┆ 8     │
│ active    ┆ 15    ┆ null       ┆ noninf       ┆ subclin       ┆ 16    │
│ active    ┆ 15    ┆ null       ┆ noninf       ┆ clin          ┆ 17    │
│ active    ┆ 15    ┆ null       ┆ inf          ┆ subclin       ┆ 18    │
│ active    ┆ 15    ┆ null       ┆ inf          ┆ clin          ┆ 19    │
│ r

In [44]:
pt = pcmap.prop_table
pt

PropertyTable
shape: (27, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ naive     ┆ 0     ┆ null       ┆ null         ┆ null          ┆ 0     │
│ naive     ┆ 15    ┆ null       ┆ null         ┆ null          ┆ 1     │
│ naive     ┆ 75    ┆ null       ┆ null         ┆ null          ┆ 2     │
│ early     ┆ 0     ┆ incipient  ┆ null         ┆ null          ┆ 3     │
│ early     ┆ 0     ┆ contained  ┆ null         ┆ null          ┆ 4     │
│ …         ┆ …     ┆ …          ┆ …            ┆ …             ┆ …     │
│ active    ┆ 75    ┆ null       ┆ inf          ┆ subclin       ┆ 22    │
│ active    ┆ 75    ┆ null       ┆ inf          ┆ clin          ┆ 23    │
│ recover

In [81]:
ahist = slp.build_astrat("history", history.prop_key.strata)

In [85]:
ahist._set_ptable(pt)

In [ ]:
age_strat = cmap.stratify_with(age_strat)
# age_strat now gets updated as an accessor

{Stratification: history: PropertyAccessor[Stratification: history],
 Stratification: age: PropertyAccessor[Stratification: age],
 Stratification: early_tb: PropertyAccessor[Stratification: early_tb],
 Stratification: inf_status: PropertyAccessor[Stratification: inf_status],
 Stratification: clin_status: PropertyAccessor[Stratification: clin_status]}

In [86]:
pt.filter(ahist == "early")

KeyError: AccessorStrat[history]

In [ ]:
srcq = acc.history=="early"
destq = acc.history=="active"

ft = pt.filter(acc.history=="early")

valid_columns(ft), invalid_columns(ft)

(['history_0', 'age_0', 'early_tb_0'], ['inf_status_0', 'clin_status_0'])

In [ ]:
pt.table

history_0,age_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64
0,0,null,null,null,0
0,1,null,null,null,1
0,2,null,null,null,2
1,0,0,null,null,3
1,0,1,null,null,4
…,…,…,…,…,…
2,2,null,1,0,22
2,2,null,1,1,23
3,0,null,null,null,24


In [ ]:
pcmap.filter(acc.history == "early"), pcmap.filter(acc.history=="active")

([PCMap]PropertyTable
 shape: (9, 6)
 ┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
 │ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
 │ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
 │ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
 ╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
 │ early     ┆ 0     ┆ incipient  ┆ null         ┆ null          ┆ 3     │
 │ early     ┆ 0     ┆ contained  ┆ null         ┆ null          ┆ 4     │
 │ early     ┆ 0     ┆ cleared    ┆ null         ┆ null          ┆ 5     │
 │ early     ┆ 15    ┆ incipient  ┆ null         ┆ null          ┆ 6     │
 │ early     ┆ 15    ┆ contained  ┆ null         ┆ null          ┆ 7     │
 │ early     ┆ 15    ┆ cleared    ┆ null         ┆ null          ┆ 8     │
 │ early     ┆ 75    ┆ incipient  ┆ null         ┆ null          ┆ 9     │
 │ early     ┆ 75    ┆ contained  ┆ null         ┆ null        

In [46]:
def qcats(strat, pt: slp.PropertyTable):
    accessor = pt.properties[strat]
    return {f"{strat.name}_{stratum}":(accessor==stratum) for stratum in strat.strata}


In [158]:
CatDict = dict[str, slp.LazyExpr]

In [169]:
ahist.ptable.filter(ahist == "15")

KeyError: AccessorStrat[history]

In [ ]:
def cat_prod(a, b):
    out_cats = {}
    for ak, av in a.items():
        for bk, bv in b.items():
            out_cats[f"{ak}__{bk}"] = av & bv
    return out_cats

class CategoryGroup:
    def __init__(self, cats: CatDict):
        self.cats = cats

    @property
    def exclusive(self) -> bool:
        

    def __mul__(self, other):
        if isinstance(other, CategoryGroup):
            return CategoryGroup(cat_prod(self.cats, other.cats))
        else:
            raise TypeError()

    def __repr__(self):
        return f"CategoryGroup:\n{self.cats}"

In [154]:
print(CategoryGroup(inf_cats) * CategoryGroup(clin_cats))

CategoryGroup:
{'inf_status_noninf__clin_status_subclin': (Stratification: inf_status == noninf & Stratification: clin_status == subclin), 'inf_status_noninf__clin_status_clin': (Stratification: inf_status == noninf & Stratification: clin_status == clin), 'inf_status_inf__clin_status_subclin': (Stratification: inf_status == inf & Stratification: clin_status == subclin), 'inf_status_inf__clin_status_clin': (Stratification: inf_status == inf & Stratification: clin_status == clin)}


In [48]:
early_cats = qcats(strats.early_tb, pt)
inf_cats = qcats(strats.inf_status, pt)
clin_cats = qcats(strats.clin_status, pt)
cat_prod(inf_cats, clin_cats)

{'inf_status_noninf__clin_status_subclin': (Stratification: inf_status == noninf & Stratification: clin_status == subclin),
 'inf_status_noninf__clin_status_clin': (Stratification: inf_status == noninf & Stratification: clin_status == clin),
 'inf_status_inf__clin_status_subclin': (Stratification: inf_status == inf & Stratification: clin_status == subclin),
 'inf_status_inf__clin_status_clin': (Stratification: inf_status == inf & Stratification: clin_status == clin)}

In [49]:
def get_cat_indices(cats: dict[str, slp.LazyExpr], pt: slp.PropertyTable):
    indices = [pt.filter(cat).table["index"].to_numpy() for cat in cats.values()]
    return np.array(indices)


In [50]:
x = np.linspace(0.0,1.0,len(pt.table))

In [61]:
pt

PropertyTable
shape: (27, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ naive     ┆ 0     ┆ null       ┆ null         ┆ null          ┆ 0     │
│ naive     ┆ 15    ┆ null       ┆ null         ┆ null          ┆ 1     │
│ naive     ┆ 75    ┆ null       ┆ null         ┆ null          ┆ 2     │
│ early     ┆ 0     ┆ incipient  ┆ null         ┆ null          ┆ 3     │
│ early     ┆ 0     ┆ contained  ┆ null         ┆ null          ┆ 4     │
│ …         ┆ …     ┆ …          ┆ …            ┆ …             ┆ …     │
│ active    ┆ 75    ┆ null       ┆ inf          ┆ subclin       ┆ 22    │
│ active    ┆ 75    ┆ null       ┆ inf          ┆ clin          ┆ 23    │
│ recover

In [69]:
from jax import numpy as jnp

In [71]:
jnp.array(x)[get_cat_indices(inf_cats,pt)]

Array([[0.46153846, 0.5       , 0.61538462, 0.65384615, 0.76923077,
        0.80769231],
       [0.53846154, 0.57692308, 0.69230769, 0.73076923, 0.84615385,
        0.88461538]], dtype=float64)

In [ ]:
pt.table[get_cat_indices(cat_prod(inf_cats, clin_cats),pt)]

TypeError: multi-dimensional NumPy arrays not supported as index

In [54]:
x[get_cat_indices(cat_prod(inf_cats, clin_cats), pt)]

array([[0.46153846, 0.61538462, 0.76923077],
       [0.5       , 0.65384615, 0.80769231],
       [0.53846154, 0.69230769, 0.84615385],
       [0.57692308, 0.73076923, 0.88461538]])

In [16]:
[pt.filter(cat).table["index"].to_numpy() for cat in early_cats.values()]

[array([1]), array([2]), array([3])]

In [17]:
app_matrix = np.zeros(3,4)

TypeError: Cannot interpret '4' as a data type

In [22]:
pop_splits = {
    inf_status.prop_key: np.array([0.8,0.2]),
    clin_status.prop_key: np.array([0.4,0.6])
}

In [19]:
pcmap.filter(history == "active")

[PCMap]PropertyTable
shape: (12, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ active    ┆ 0     ┆ null       ┆ noninf       ┆ subclin       ┆ 12    │
│ active    ┆ 0     ┆ null       ┆ noninf       ┆ clin          ┆ 13    │
│ active    ┆ 0     ┆ null       ┆ inf          ┆ subclin       ┆ 14    │
│ active    ┆ 0     ┆ null       ┆ inf          ┆ clin          ┆ 15    │
│ active    ┆ 15    ┆ null       ┆ noninf       ┆ subclin       ┆ 16    │
│ …         ┆ …     ┆ …          ┆ …            ┆ …             ┆ …     │
│ active    ┆ 15    ┆ null       ┆ inf          ┆ clin          ┆ 19    │
│ active    ┆ 75    ┆ null       ┆ noninf       ┆ subclin       ┆ 20    │
│ 

In [20]:
early_tb.prop_key.strata

('incipient', 'contained', 'cleared')

In [23]:
early_tb.prop_key.categories(), inf_status.prop_key.categories()

(CategoryGroup
 Category: [(Stratification: early_tb, ['incipient'])]
 Category: [(Stratification: early_tb, ['contained'])]
 Category: [(Stratification: early_tb, ['cleared'])],
 CategoryGroup
 Category: [(Stratification: inf_status, ['noninf'])]
 Category: [(Stratification: inf_status, ['inf'])])

In [24]:
pcmap.filter(history == "early")

[PCMap]PropertyTable
shape: (9, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ early     ┆ 0     ┆ incipient  ┆ null         ┆ null          ┆ 0     │
│ early     ┆ 0     ┆ contained  ┆ null         ┆ null          ┆ 1     │
│ early     ┆ 0     ┆ cleared    ┆ null         ┆ null          ┆ 2     │
│ early     ┆ 15    ┆ incipient  ┆ null         ┆ null          ┆ 3     │
│ early     ┆ 15    ┆ contained  ┆ null         ┆ null          ┆ 4     │
│ early     ┆ 15    ┆ cleared    ┆ null         ┆ null          ┆ 5     │
│ early     ┆ 75    ┆ incipient  ┆ null         ┆ null          ┆ 6     │
│ early     ┆ 75    ┆ contained  ┆ null         ┆ null          ┆ 7     │
│ e

In [18]:
pcmap.filter(history == "active")

[PCMap]PropertyTable
shape: (12, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ active    ┆ 0     ┆ null       ┆ noninf       ┆ subclin       ┆ 0     │
│ active    ┆ 0     ┆ null       ┆ noninf       ┆ clin          ┆ 1     │
│ active    ┆ 0     ┆ null       ┆ inf          ┆ subclin       ┆ 2     │
│ active    ┆ 0     ┆ null       ┆ inf          ┆ clin          ┆ 3     │
│ active    ┆ 15    ┆ null       ┆ noninf       ┆ subclin       ┆ 4     │
│ …         ┆ …     ┆ …          ┆ …            ┆ …             ┆ …     │
│ active    ┆ 15    ┆ null       ┆ inf          ┆ clin          ┆ 7     │
│ active    ┆ 75    ┆ null       ┆ noninf       ┆ subclin       ┆ 8     │
│ 

In [16]:
pcmap.filter(history.is_between("early","active") & (age=="15"))

[PCMap]PropertyTable
shape: (7, 6)
┌───────────┬───────┬────────────┬──────────────┬───────────────┬───────┐
│ history_0 ┆ age_0 ┆ early_tb_0 ┆ inf_status_0 ┆ clin_status_0 ┆ index │
│ ---       ┆ ---   ┆ ---        ┆ ---          ┆ ---           ┆ ---   │
│ str       ┆ str   ┆ str        ┆ str          ┆ str           ┆ i64   │
╞═══════════╪═══════╪════════════╪══════════════╪═══════════════╪═══════╡
│ early     ┆ 15    ┆ incipient  ┆ null         ┆ null          ┆ 0     │
│ early     ┆ 15    ┆ contained  ┆ null         ┆ null          ┆ 1     │
│ early     ┆ 15    ┆ cleared    ┆ null         ┆ null          ┆ 2     │
│ active    ┆ 15    ┆ null       ┆ noninf       ┆ subclin       ┆ 3     │
│ active    ┆ 15    ┆ null       ┆ noninf       ┆ clin          ┆ 4     │
│ active    ┆ 15    ┆ null       ┆ inf          ┆ subclin       ┆ 5     │
│ active    ┆ 15    ┆ null       ┆ inf          ┆ clin          ┆ 6     │
└───────────┴───────┴────────────┴──────────────┴───────────────┴───────┘

In [44]:
age.prop_key.strata

('0', '15', '75')

In [42]:
pcmap.filter(age < "15")

In [39]:
class DerivedAccessKey:
    def __init__(self, k, modifier):
        self.k = k
        self.modifier = modifier

    def __hash__(self):
        return hash((self.k,self.modifier))
    
    def __repr__(self):
        return f"{self.modifier}[{self.k}]"

In [41]:
DerivedAccessKey(history, "dest")

dest[PropertyAccessor[Stratification: history]]

In [9]:
pcmap.prop_table.properties

{Stratification: history: PropertyAccessor[Stratification: history],
 Stratification: age: PropertyAccessor[Stratification: age],
 Stratification: early_tb: PropertyAccessor[Stratification: early_tb],
 Stratification: inf_status: PropertyAccessor[Stratification: inf_status],
 Stratification: clin_status: PropertyAccessor[Stratification: clin_status]}

In [ ]:
pcmap.prop_table.table.with_columns(index=np.arange(5,len(pcmap.prop_table.table)+5))

history_0,age_0,early_tb_0,inf_status_0,clin_status_0,index
i64,i64,i64,i64,i64,i64
0,0,null,null,null,5
0,1,null,null,null,6
0,2,null,null,null,7
1,0,0,null,null,8
1,0,1,null,null,9
…,…,…,…,…,…
2,2,null,1,0,27
2,2,null,1,1,28
3,0,null,null,null,29


In [ ]:
((age == "15") or (history == "naive"))

In [ ]:
from bidict import bidict

In [ ]:
from typing import Hashable

UnameIndexKeyMap = dict[str, bidict[int, str]]
UnameObjectMap = dict[str, Hashable]


In [ ]:
pcmap.prop_table.uname_strat_map

bidict({'history_0': Stratification: history, 'age_0': Stratification: age, 'early_tb_0': Stratification: early_tb, 'inf_status_0': Stratification: inf_status, 'clin_status_0': Stratification: clin_status})

In [ ]:
pcmap.filter((age=='0') & (history == "naive")).prop_table

history_0,age_0,index
i64,i64,i64
0,0,0


In [ ]:
pcmap.filter(((age == "15") & ((inf_status == "noninf") | (inf_status.is_null())))).cmap

CompartmentContainer view of 0x2206253922832:
array([Compartment :[(Stratification: history, 'naive'), (Stratification: age, '15')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'incipient')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'contained')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'cleared')],
       Compartment :[(Stratification: history, 'active'), (Stratification: age, '15'), (Stratification: inf_status, 'noninf'), (Stratification: clin_status, 'subclin')],
       Compartment :[(Stratification: history, 'active'), (Stratification: age, '15'), (Stratification: inf_status, 'noninf'), (Stratification: clin_status, 'clin')],
       Compartment :[(Stratification: history, 'recovery'), (Stratification: age, '15')]],
      dtype=object)array([ 1,  6,  7,  8, 16, 17, 25])

In [ ]:
age.prop_key.categories()

CategoryGroup
Category: [(Stratification: age, ['0'])]
Category: [(Stratification: age, ['15'])]
Category: [(Stratification: age, ['75'])]

In [ ]:
age.prop_key.strata[1:]

('15', '75')

In [ ]:
def filter_strat(astrat, expr):
    ftable = astrat.ptable.table.filter(expr.actualize(astrat.ptable))
    slp.AccessorStrat()
    strat = expr.pa.prop_key
    return strat

In [ ]:
astrat = slp.build_astrat("age", pl.Series([str(i) for i in range(32768)]))

In [ ]:
astrat.ptable.table.filter((astrat > "15").actualize(astrat.ptable))

age_0,index
i64,i64
16,16
17,17
18,18
19,19
20,20
…,…
32763,32763
32764,32764
32765,32765


In [ ]:
pl.Series(["0","15"])

""
str
"""0"""
"""15"""


In [ ]:
class NewStrat:
    def __init__(self, name, strata, table):
        self.name = name
        self.strata = strata
        self.table = table

    @classmethod
    def new(cls, name, strata):
        strat = cls(name, strata, None)
        table = slp.strat_to_prop_table(strat)
        strat.table = table


In [ ]:
agept = slp.strat_to_prop_table(age.prop_key)
accessor = age #list(agept.properties.values())[0]

In [ ]:
def filter_pt(pt, expr):
    query_expr = expr.actualize(pt)
    fpt = pt.table.filter(query_expr)

    validity_s = fpt.select(~pl.all().eq(-1).all())

    valid_columns = [c.name for c in validity_s if c[0]]
    #valid_columns = [k for k,v in validity.items() if v]

    valid_pt = fpt[valid_columns]

    return valid_pt

In [ ]:
age.prop_key.strata[(0,1)]

TypeError: tuple indices must be integers or slices, not tuple

In [ ]:
age.prop_key.strata[filter_pt(agept, accessor >= "15")["index"].to_list()]

TypeError: tuple indices must be integers or slices, not list

In [ ]:
(age >= "15").actualize(agept)["index"]

TypeError: 'Expr' object is not subscriptable

In [ ]:
pcmap.filter((age >= "15") & (history > "naive")).cmap

CompartmentContainer view of 0x1599914834448:
array([Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'incipient')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'contained')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '15'), (Stratification: early_tb, 'cleared')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '75'), (Stratification: early_tb, 'incipient')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '75'), (Stratification: early_tb, 'contained')],
       Compartment :[(Stratification: history, 'early'), (Stratification: age, '75'), (Stratification: early_tb, 'cleared')],
       Compartment :[(Stratification: history, 'active'), (Stratification: age, '15'), (Stratification: inf_status, 'noninf'), (Stratification: clin_status, 'subclin')],
       Compartment :

In [ ]:
pt.properties

{Stratification: history: PropertyAccessor[Stratification: history],
 Stratification: age: PropertyAccessor[Stratification: age],
 Stratification: early_tb: PropertyAccessor[Stratification: early_tb],
 Stratification: inf_status: PropertyAccessor[Stratification: inf_status],
 Stratification: clin_status: PropertyAccessor[Stratification: clin_status]}

In [ ]:
type((age > "15").actualize(pt))

polars.expr.expr.Expr

In [ ]:
query = (age != "15") & (history == "naive")

query_expr = query.actualize(pt)
fpt = pt.table.filter(query_expr)

validity_s = fpt.select(~pl.all().eq(-1).all())
validity = {c.name:c[0] for c in validity_s}

valid_columns = [c.name for c in validity_s if c[0]]
#valid_columns = [k for k,v in validity.items() if v]

fpt[valid_columns]

history_0,age_0,index
i64,i64,i64
0,0,0
0,2,2


In [ ]:
history.prop_key.strata

('naive', 'early', 'active', 'recovery')

In [ ]:
query = ~(inf_status == "inf") & (history == "recovery")

query_expr = query.actualize(pt)
fpt = pt.table.filter(query_expr)

validity_s = fpt.select(~pl.all().eq(-1).all())
validity = {c.name:c[0] for c in validity_s}

valid_columns = [c.name for c in validity_s if c[0]]
#valid_columns = [k for k,v in validity.items() if v]

fpt[valid_columns]

history_0,age_0,index
i64,i64,i64
3,0,24
3,1,25
3,2,26
